In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

for tabela in [
    "tickets",
    "users",
    "tickets_users",
    "ticketsatisfactions",
]:
    spark.table(f"{catalog}.{schema}.silver_{tabela}") \
         .createOrReplaceTempView(f"silver_{tabela}")

    print(f"Extraindo tabela {tabela}")

In [0]:
d_users = spark.sql("""
SELECT DISTINCT

    user_id,
    login,
    nome_completo

FROM silver_users

WHERE user_id IS NOT NULL
""")

d_status = spark.sql("""
SELECT DISTINCT

    status_id,
    status_descricao

FROM silver_tickets
""")

p_tickets_users = spark.sql("""
SELECT
    ticket_id,
    user_id      AS usuario_id,
    CASE type
        WHEN 1 THEN 'Solicitante'
        WHEN 2 THEN 'Técnico'
        WHEN 3 THEN 'Grupo responsável'
        ELSE 'Outro'
END AS tipo_relacionamento

FROM silver_tickets_users
WHERE user_id IS NOT NULL
""")

f_tickets = spark.sql("""
SELECT
    t.ticket_id,
    t.itilcategories_id,
    t.entities_id,
    t.status_id,
    t.priority,
    t.urgency,
    t.impact,
    t.data_criacao,
    t.data_abertura,
    t.data_modificacao,
    t.data_solucao,
    t.data_fechamento,
    t.horas_resolucao,
    t.deletado,
    t.titulo,
    t.content
FROM silver_tickets t
""")

f_tickets.createOrReplaceTempView("f_tickets")

f_ticketsatisfactions = spark.sql("""
SELECT

    id                      AS satisfacao_id,
    tickets_id              AS ticket_id,

    satisfaction,

    comment,

    date_begin,
    date_answered

FROM silver_ticketsatisfactions s
                           
INNER JOIN f_tickets t
    ON s.tickets_id = t.ticket_id
""")

def save_tabelas(df, tabela):
    df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{tabela}")

tabelas_map = [
    (f_tickets, "f_tickets"),
    (f_ticketsatisfactions, "f_ticketsatisfactions"),
    (d_users, "d_users"),
    (d_status, "d_status"),
    (p_tickets_users, "p_tickets_users"),
]

for df, nome_tabela in tabelas_map:
    save_tabelas(df, nome_tabela)
    print(f"{nome_tabela} criada")